# Proyecto 2: Big Data - Análisis Descriptivo con PySpark
**Asignatura:** SI3007 Sistemas Distribuidos / ST0263 Tópicos Especiales en Telemática
**Tema:** Análisis de Natalidad en Colombia (2021-2025) y Contexto Internacional (Banco Mundial)

En este notebook se realiza el procesamiento analítico y descriptivo utilizando **Apache Spark (PySpark)** sobre los datos procesados en la zona `trusted` (formato Parquet) almacenados en Amazon S3.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Inicialización de la sesión de Spark
spark = SparkSession.builder \
    .appName('Analisis-Natalidad-PySpark') \
    .getOrCreate()

BUCKET = 'bigdata-p2-190565384867'
BASE_S3 = f's3://{BUCKET}'
print('SparkSession iniciada con éxito')

## 1. Carga y Normalización de Datos desde la Zona Trusted de S3

In [2]:
df_nacimientos = spark.read.parquet(f'{BASE_S3}/trusted/nacimientos/')
df_wb = spark.read.parquet(f'{BASE_S3}/trusted/world_bank/')
df_deptos = spark.read.parquet(f'{BASE_S3}/trusted/rds/dim_departamento/')
df_catalogo = spark.read.parquet(f'{BASE_S3}/trusted/rds/dim_catalogo/')

# Normalización de columnas a minúsculas
df_nacimientos = df_nacimientos.toDF(*[c.lower() for c in df_nacimientos.columns])
df_deptos = df_deptos.toDF(*[c.lower() for c in df_deptos.columns])
df_catalogo = df_catalogo.toDF(*[c.lower() for c in df_catalogo.columns])
df_wb = df_wb.toDF(*[c.lower() for c in df_wb.columns])

print('Total registros en nacimientos:', df_nacimientos.count())
df_deptos.cache()
df_catalogo.cache()

## 2. Estadísticas Descriptivas Generales

In [3]:
vars_num = [c for c in ['numconsul', 'edad_padre', 'apgar1', 'apgar2', 'n_hijosv', 'n_emb'] if c in df_nacimientos.columns]
df_nacimientos.select(vars_num).describe().show()

## 3. Pregunta 1: Evolución por Año y Departamento (2021-2025)

In [4]:
p1 = (df_nacimientos
      .join(df_deptos, on='cod_dpto', how='left')
      .groupBy(F.col('anio'), F.coalesce(F.col('nombre'), F.col('cod_dpto')).alias('departamento'))
      .agg(F.count(F.lit(1)).alias('total_nacimientos'))
      .orderBy('anio', F.desc('total_nacimientos')))
p1.show(15, truncate=False)

## 4. Pregunta 2: Estacionalidad de Nacimientos por Mes

In [5]:
p2 = (df_nacimientos
      .filter(F.col('mes').isNotNull())
      .groupBy('mes')
      .agg(F.count(F.lit(1)).alias('total_nacimientos'),
           F.round(F.count(F.lit(1)) / 5.0, 0).alias('promedio_anual'))
      .orderBy(F.desc('total_nacimientos')))
p2.show(12)

## 5. Pregunta 3: Madres Adolescentes por Departamento

In [6]:
p3 = (df_nacimientos
      .filter(F.col('edad_madre').isNotNull())
      .join(df_deptos, on='cod_dpto', how='left')
      .groupBy(F.coalesce(F.col('nombre'), F.col('cod_dpto')).alias('departamento'))
      .agg(
          F.count(F.lit(1)).alias('total_nacimientos'),
          F.sum(F.when(F.col('edad_madre').isin(['1', '2']), 1).otherwise(0)).alias('nacimientos_adolescentes')
      )
      .withColumn('pct_madres_adolescentes', F.round(100.0 * F.col('nacimientos_adolescentes') / F.col('total_nacimientos'), 2))
      .orderBy(F.desc('pct_madres_adolescentes')))
p3.show(15, truncate=False)

## 6. Pregunta 4: Nivel Educativo vs Controles y Cesáreas

In [7]:
cat_edu = df_catalogo.filter(F.col('variable') == 'NIV_EDUM')
p4 = (df_nacimientos
      .filter(F.col('niv_edum').isNotNull())
      .withColumn('cod_edu_pad', F.lpad(F.col('niv_edum'), 2, '0'))
      .join(cat_edu, F.col('cod_edu_pad') == F.col('codigo'), 'left')
      .groupBy(F.coalesce(F.col('descripcion'), F.lit('Otro / No especificado')).alias('nivel_educativo'))
      .agg(
          F.count(F.lit(1)).alias('total_nacimientos'),
          F.round(F.avg('numconsul'), 2).alias('promedio_controles'),
          F.round(100.0 * F.sum(F.when(F.col('tipo_parto') == '2', 1).otherwise(0)) / F.count(F.lit(1)), 2).alias('pct_cesareas')
      )
      .orderBy(F.desc('promedio_controles')))
p4.show(15, truncate=False)

## 7. Pregunta 5: Comparativa Internacional (Banco Mundial)

In [8]:
p5 = (df_wb
      .filter(F.col('indicador') == 'WB_WDI_SP_DYN_CBRT_IN')
      .select('pais', 'anio', F.round('valor', 2).alias('tasa_bruta_natalidad'))
      .orderBy(F.desc('anio'), F.desc('tasa_bruta_natalidad')))
p5.show(15, truncate=False)